In [1]:
import pandas as pd
import joblib

# Load the saved tuned model
best_lr_model = joblib.load('final_model.pkl')

# Load test data fresh
X_test = pd.read_csv('../Data/X_test.csv')
y_test = pd.read_csv('../Data/y_test.csv')
y_test = y_test['payment_recovered']  # flatten to Series, same fix as before

print(type(best_lr_model))
print(X_test.shape, y_test.shape)

<class 'sklearn.linear_model._logistic.LogisticRegression'>
(1409, 17) (1409,)


In [2]:
y_proba_tuned = best_lr_model.predict_proba(X_test)[:, 1]
print(y_proba_tuned[:5])

[0.50060589 0.60378411 0.84671656 0.60994226 0.08793912]


In [3]:
def recommend_action(probability, failure_reason, customer_segment, retry_count):
    
    if failure_reason == 'card_expired':
        action = 'switch_payment_method'
        explanation = f"Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead."
    
    elif failure_reason == 'insufficient_funds' and probability >= 0.4:
        action = 'delayed_retry'
        explanation = f"Failure reason is insufficient_funds with a reasonable recovery probability ({probability:.2f}). Waiting allows time for the customer's balance to potentially refill before retrying."
    
    elif probability >= 0.7:
        action = 'immediate_retry'
        explanation = f"High recovery probability ({probability:.2f}) with no time-dependent failure reason. Retrying immediately is likely to succeed."
    
    elif probability < 0.4 and customer_segment == 'high_value_repeat':
        action = 'send_incentive'
        explanation = f"Low recovery probability ({probability:.2f}), but customer is high-value and repeat. Worth offering an incentive to actively recover this payment."
    
    else:
        action = 'delayed_retry'
        explanation = f"No strong signal for immediate action (probability={probability:.2f}). Defaulting to a low-cost delayed retry rather than an expensive incentive."
    
    return action, explanation

In [4]:
print(recommend_action(0.9, 'card_expired', 'occasional', 1))
print(recommend_action(0.8, 'insufficient_funds', 'occasional', 0))
print(recommend_action(0.85, 'network_error', 'new_customer', 0))
print(recommend_action(0.25, 'bank_decline', 'high_value_repeat', 1))
print(recommend_action(0.2, 'bank_decline', 'new_customer', 2))

('switch_payment_method', 'Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead.')
('delayed_retry', "Failure reason is insufficient_funds with a reasonable recovery probability (0.80). Waiting allows time for the customer's balance to potentially refill before retrying.")
('immediate_retry', 'High recovery probability (0.85) with no time-dependent failure reason. Retrying immediately is likely to succeed.')
('send_incentive', 'Low recovery probability (0.25), but customer is high-value and repeat. Worth offering an incentive to actively recover this payment.')
('delayed_retry', 'No strong signal for immediate action (probability=0.20). Defaulting to a low-cost delayed retry rather than an expensive incentive.')


In [5]:
segment_cols = ['customer_segment_high_value_repeat', 
                'customer_segment_new_customer', 
                'customer_segment_occasional']

X_test['customer_segment_decoded'] = X_test[segment_cols].idxmax(axis=1)
X_test['customer_segment_decoded'] = X_test['customer_segment_decoded'].str.replace('customer_segment_', '')

print(X_test['customer_segment_decoded'].value_counts())

customer_segment_decoded
occasional           674
high_value_repeat    494
new_customer         241
Name: count, dtype: int64


In [8]:
failure_cols = ['failure_reason_bank_decline', 'failure_reason_card_expired', 
                'failure_reason_insufficient_funds', 'failure_reason_network_error']

X_test['failure_reason_decoded'] = X_test[failure_cols].idxmax(axis=1)
X_test['failure_reason_decoded'] = X_test['failure_reason_decoded'].str.replace('failure_reason_', '')

print(X_test['failure_reason_decoded'].value_counts())

failure_reason_decoded
insufficient_funds    466
card_expired          350
bank_decline          322
network_error         271
Name: count, dtype: int64


In [9]:
def apply_agent(row):
    action, explanation = recommend_action(
        row['probability'],
        row['failure_reason_decoded'],
        row['customer_segment_decoded'],
        row['retry_count']
    )
    return pd.Series([action, explanation])

X_test[['recommended_action', 'explanation']] = X_test.apply(apply_agent, axis=1)

print(X_test['recommended_action'].value_counts())

recommended_action
delayed_retry            792
switch_payment_method    350
immediate_retry          266
send_incentive             1
Name: count, dtype: int64


In [ ]:
def recommend_action(probability, failure_reason, customer_segment, retry_count):
    
    if failure_reason == 'card_expired':
        action = 'switch_payment_method'
        explanation = f"Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead."
    
    elif failure_reason == 'insufficient_funds' and probability >= 0.4:
        action = 'delayed_retry'
        explanation = f"Failure reason is insufficient_funds with a reasonable recovery probability ({probability:.2f}). Waiting allows time for the customer's balance to potentially refill before retrying."
    
    elif probability >= 0.7:
        action = 'immediate_retry'
        explanation = f"High recovery probability ({probability:.2f}) with no time-dependent failure reason. Retrying immediately is likely to succeed."
    
    elif probability < 0.5 and customer_segment == 'high_value_repeat':
        action = 'send_incentive'
        explanation = f"Recovery probability is moderate-to-low ({probability:.2f}), but customer is high-value and repeat. Worth offering an incentive to actively recover this payment."
    
    else:
        action = 'delayed_retry'
        explanation = f"No strong signal for immediate action (probability={probability:.2f}). Defaulting to a low-cost delayed retry rather than an expensive incentive."
    
    return action, explanation

In [11]:
X_test[['recommended_action', 'explanation']] = X_test.apply(apply_agent, axis=1)
print(X_test['recommended_action'].value_counts())

recommended_action
delayed_retry            781
switch_payment_method    350
immediate_retry          266
send_incentive            12
Name: count, dtype: int64


In [12]:
pd.set_option('display.max_colwidth', None)
X_test[['probability', 'failure_reason_decoded', 'customer_segment_decoded', 'retry_count', 'recommended_action', 'explanation']].sample(5)

,probability,failure_reason_decoded,customer_segment_decoded,retry_count,recommended_action,explanation
963,0.867155,network_error,occasional,0,immediate_retry,High recovery probability (0.87) with no time-dependent failure reason. Retrying immediately is likely to succeed.
146,0.086689,card_expired,occasional,0,switch_payment_method,"Failure reason is card_expired, which almost never recovers through retries. Recommending the customer update their payment method instead."
991,0.792550,network_error,occasional,1,immediate_retry,High recovery probability (0.79) with no time-dependent failure reason. Retrying immediately is likely to succeed.
583,0.476742,bank_decline,occasional,1,delayed_retry,No strong signal for immediate action (probability=0.48). Defaulting to a low-cost delayed retry rather than an expensive incentive.
1377,0.601667,bank_decline,new_customer,0,delayed_retry,No strong signal for immediate action (probability=0.60). Defaulting to a low-cost delayed retry rather than an expensive incentive.


In [13]:
X_test.to_csv('../Data/test_with_recommendations.csv', index=False)
print("Saved successfully")

Saved successfully
